## Overview

The purest way of representing date-time is the timestamp, which is the number of seconds elapsed since the **epoch**. The **epoch** is midnight of 1 Jan, 1970 UTC. The timestamp has no timezone, the epoch does. For example in Pacific Time the epoch is 4PM of 31 Dec, 1969, but the number of seconds elapsed since then is the same as the number of seconds elapsed since midnight of 1 Jan, 1970 UTC. However, when I want to represent the timestamp in terms of year, month, day, hours, etc. then I need some timezone to anchor it. By default Python has what it calls the "naive" datetime, this is also referred to as calendar date and wall clock time, it is the datetime I'd read off a calendar or a clock, it won't come with a timezone attached. Python will use the host's locale to report the year, month, day, hour, etc.

There are two types of timezone conversions I might want -

- A new datetime object which has the same year, month, day, hour, etc. as the original one, but a different timezone attached to it. For example, Jul 19, 2025 12:47PM PT will become Jul 19, 2025 12:47PM IST. For this to happen the timestamp will have to change.
- A new datetime object which converts the original into a different timezone. For example, Jul 19, 2025 12:47PM PT will become Jul 20, 2025 01:17 AM IST. Here the timestamp remains the same, only the representation changes.


## Formatting


Some of my go-to datetime formats are -

- 2025-01-30 => `%Y-%m-%d`
- Thu, Jan 1, 2025 => `%a, %b %d, %Y`
- 11:30 AM => `%I:%M %p`

Common format codes that I end up using -

- `%Y` Year with century as a zero-padded decimal number
- `%m` Month as zero-padded decimal number
- `%b` Month as locale's abbreviated name
- `%B` Month as locale's full name

- `%d` Day of the month as a zero-padded decimal number
- `%a` Weekday as locale's abbreviated name
- `%A` Weekday as locale's full name

- `%I` Hour (12-hour) clock as a zero-padded decimal number
- `%M` Minute as zero-padded decimal number
- `%S` Seconda as a zero-padded decimal number
- `%p` Locale's equivalent of either AM or PM


In [1]:
%reset -f
from datetime import datetime

In [2]:
datetime.strptime("2025-01-30", "%Y-%m-%d")

datetime.datetime(2025, 1, 30, 0, 0)

In [3]:
datetime.strptime("Thu, Jan 30, 2025", "%a, %b %d, %Y")

datetime.datetime(2025, 1, 30, 0, 0)

In [4]:
datetime.strptime("11:30 AM", "%I:%M %p")

datetime.datetime(1900, 1, 1, 11, 30)

In [5]:
dt = datetime(year=2025, month=1, day=30, hour=11, minute=30)
dt.strftime("%Y-%m-%d, %a %I:%M %p")

'2025-01-30, Thu 11:30 AM'

In [6]:
dt.isoformat()

'2025-01-30T11:30:00'

In [7]:
dt.isoformat(timespec="minutes")

'2025-01-30T11:30'

## Timezones


In [8]:
%reset -f
from datetime import datetime, timezone
import pytz

In [9]:
for tz in pytz.common_timezones:
    print(tz)

Africa/Abidjan
Africa/Accra
Africa/Addis_Ababa
Africa/Algiers
Africa/Asmara
Africa/Bamako
Africa/Bangui
Africa/Banjul
Africa/Bissau
Africa/Blantyre
Africa/Brazzaville
Africa/Bujumbura
Africa/Cairo
Africa/Casablanca
Africa/Ceuta
Africa/Conakry
Africa/Dakar
Africa/Dar_es_Salaam
Africa/Djibouti
Africa/Douala
Africa/El_Aaiun
Africa/Freetown
Africa/Gaborone
Africa/Harare
Africa/Johannesburg
Africa/Juba
Africa/Kampala
Africa/Khartoum
Africa/Kigali
Africa/Kinshasa
Africa/Lagos
Africa/Libreville
Africa/Lome
Africa/Luanda
Africa/Lubumbashi
Africa/Lusaka
Africa/Malabo
Africa/Maputo
Africa/Maseru
Africa/Mbabane
Africa/Mogadishu
Africa/Monrovia
Africa/Nairobi
Africa/Ndjamena
Africa/Niamey
Africa/Nouakchott
Africa/Ouagadougou
Africa/Porto-Novo
Africa/Sao_Tome
Africa/Tripoli
Africa/Tunis
Africa/Windhoek
America/Adak
America/Anchorage
America/Anguilla
America/Antigua
America/Araguaina
America/Argentina/Buenos_Aires
America/Argentina/Catamarca
America/Argentina/Cordoba
America/Argentina/Jujuy
America/

In [10]:
fmt = "%a, %b %d, %Y | %I:%M %p %Z %z"

In [11]:
pt = pytz.timezone("US/Pacific")
ist = pytz.timezone("Asia/Kolkata")
utc = pytz.utc
print(pt.zone, ist.zone, utc.zone)

US/Pacific Asia/Kolkata UTC


As a general rule, I should always use UTC in all my programs, except when I have to display the time to the user. At that point in my program I can conver the UTC time to local time and display. This is because it is tricky to construct local time and do datetime arithmetic on local times as described in [Problems with Localtime](https://pythonhosted.org/pytz/#problems-with-localtime).

There are two ways I can get local datetime -

- Using the `localize` method which changes the timestamp but keeps the original representation, i.e., colloquially speaking it will "attach" a timezone to a datetime.
- Using the `astimezone` method which keeps the original timestamp but changes the representation, i.e., colloquially speaking it will "convert" the datetime to a new timezone.

> 💣 Using the `tzinfo` param in the datetime constructor will not work for timezones with daylight savings!

> 💣 Using `datetime.replace` does not work intuitively to attach timezone.


In [12]:
# The default now method has no timezone attached to it, but it gives the local time.
now = datetime.now()
print(f"now: {now.strftime(fmt)}\ntzinfo: {now.tzinfo}\ntimestamp: {now.timestamp():,}")

now: Sat, Jul 19, 2025 | 12:51 PM  
tzinfo: None
timestamp: 1,752,954,691.514354


In [ ]:
# !!! DO NOT USE replace !!!
# I probably don't understand how to use pytz and datetime together in this method, but it does not
# match intuition. Some things that don't match -
#   - Why is the timezone +553 instead of +530?
#   - Why is the timezone format specifier %Z not working correctly, it is still showing LMT?
ist_dt = now.replace(tzinfo=ist)
print(
    f"ist_dt: {ist_dt.strftime(fmt)}\ntzinfo: {ist_dt.tzinfo}\ntimestamp: {ist_dt.timestamp():,}"
)

ist_dt: Sat, Jul 19, 2025 | 12:51 PM LMT +0553
tzinfo: Asia/Kolkata
timestamp: 1,752,908,311.514354


In [ ]:
# The effect is that the year, month, day, hour, minute, second, etc. remain the "same" with only
# the timezone attached to the object. Internally, the timestamp was changed s.t. the day/hour/etc.
# would match the new timezone. Notice that the IST timestamp is less than local (PT) timestamp,
# because 5:15PM happened earlier in IST than in PT.
ist_dt_2 = ist.localize(now)
print(
    f"ist_dt_2: {ist_dt_2.strftime(fmt)}\ntzinfo: {ist_dt_2.tzinfo}\ntimestamp: {ist_dt_2.timestamp():,}"
)

ist_dt_2: Sat, Jul 19, 2025 | 12:51 PM IST +0530
tzinfo: Asia/Kolkata
timestamp: 1,752,909,691.514354


In [15]:
# Same demo as above, but with UTC instead of IST.
utc_dt = utc.localize(now)
print(
    f"utc_dt: {utc_dt.strftime(fmt)}\ntzinfo: {utc_dt.tzinfo}\ntimestamp: {utc_dt.timestamp():,}"
)

utc_dt: Sat, Jul 19, 2025 | 12:51 PM UTC +0000
tzinfo: UTC
timestamp: 1,752,929,491.514354


In [16]:
# Localizing to PT does not change the timestamp, because that is the local time.
pt_now = pt.localize(now)
print(
    f"pt_now: {pt_now.strftime(fmt)}\ntzinfo: {pt_now.tzinfo}\ntimestamp: {pt_now.timestamp():,}"
)

pt_now: Sat, Jul 19, 2025 | 12:51 PM PDT -0700
tzinfo: US/Pacific
timestamp: 1,752,954,691.514354


In [ ]:
# If I want to convert the datetime from one timezone to another, what I **really** want is to keep
# the timestamp same! Just want the year, month, day, etc. to change to the new timezone.
ist_now = pt_now.astimezone(ist)
print(
    f"ist_now: {ist_now.strftime(fmt)}\ntzinfo: {ist_now.tzinfo}\ntimestamp: {ist_now.timestamp():,}"
)

ist_now: Sun, Jul 20, 2025 | 01:21 AM IST +0530
tzinfo: Asia/Kolkata
timestamp: 1,752,954,691.514354


In [18]:
# Will also work with naive datetime objects because it is just taking the current timestamp and
# figuring out the year/month/day/hour/etc. for the given timezone.
ist_now_2 = now.astimezone(ist)
print(
    f"ist_now_2: {ist_now_2.strftime(fmt)}\ntzinfo: {ist_now_2.tzinfo}\ntimestamp: {ist_now_2.timestamp():,}"
)

ist_now_2: Sun, Jul 20, 2025 | 01:21 AM IST +0530
tzinfo: Asia/Kolkata
timestamp: 1,752,954,691.514354


In [19]:
# The tzinfo param in ctor does not work - the timezone is LMT!
dt = datetime(year=2025, month=6, day=26, hour=18, minute=11, tzinfo=pt)
dt.strftime(fmt)

'Thu, Jun 26, 2025 | 06:11 PM LMT -0753'

In [20]:
# Works with UTC which has no daylight savings.
dt = datetime(year=2025, month=6, day=26, hour=18, minute=11, tzinfo=utc)
dt.strftime(fmt)

'Thu, Jun 26, 2025 | 06:11 PM UTC +0000'

In [21]:
# Should've worked with IST which has no daylight savings either, but doesn't!
dt = datetime(year=2025, month=6, day=26, hour=18, minute=11, tzinfo=ist)
dt.strftime(fmt)

'Thu, Jun 26, 2025 | 06:11 PM LMT +0553'

In [22]:
# Works with builtin UTC timezone which has no daylight savings.
dt = datetime(year=2025, month=6, day=26, hour=18, minute=11, tzinfo=timezone.utc)
dt.strftime(fmt)

'Thu, Jun 26, 2025 | 06:11 PM UTC +0000'